# Teste isolado — ONS Notícias (Fase 1)

Fonte candidata: **ONS - Notícias** (Operador Nacional do Sistema Elétrico), setor Energia.
Listagem: `https://www.ons.org.br/paginas/imprensa/noticias`

Notebook **descartável**, conforme a Fase 1 do fluxo de adição de fonte nova: não depende de
nenhum dispatcher, não chama `atualizar_status_fonte`, não grava nada no Volume nem em tabela
de controle. Só valida duas coisas e imprime amostra para conferência manual:

1. Listar as notícias (data + título + link da página de detalhe)
2. Abrir uma notícia individual e extrair o texto completo

## Duas descobertas que mudam a implementação

**1. O host precisa do `www`.** `https://ons.org.br/...` devolve **404** — só
`https://www.ons.org.br/...` responde. (O `robots.txt` também não existe: 404. Não há
`Disallow` a respeitar, diferente do caso da AGENERSA.)

**2. A página não serve HTML — nem a listagem, nem o detalhe.** O site é SharePoint com
renderização client-side: o HTML cru da listagem **não contém nenhum link `details.aspx`**, e o
HTML do detalhe não contém o texto da notícia. Raspar com BeautifulSoup em cima do HTML devolve
zero itens.

O conteúdo vem de dois endpoints REST públicos (sem token), descobertos lendo os JS do próprio
site — `/Style Library/custom/js/wp_noticias.js` e `wp_noticiasDetalhe.js`:

| O quê | Endpoint |
|---|---|
| Listagem (paginada) | `…/attachment/_api/web/lists/getbytitle('Multicanais - Notícias')/items` |
| Texto da notícia | `…/api/noticias/get?id={Id}` |

Ambos em `https://proxyportais.ons.org.br/ons.portalempregado.proxy/`.

**Consequência prática: não precisa de Selenium.** Apesar de ser um site JS, dá para consumir
com `httpx` puro — ou seja, esta fonte *não* cai no bloqueio de Chrome/init script que trava
Brazil Journal e PPI hoje.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import json
import time
import random
import urllib.parse
from datetime import datetime
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

# Página pública da listagem — serve só de referência; o HTML dela é uma casca
# vazia (ver observação no topo do notebook).
SITE_URL = "https://www.ons.org.br/paginas/imprensa/noticias"

# Proxy REST que o próprio site consome via fetch().
PROXY_BASE = "https://proxyportais.ons.org.br/ons.portalempregado.proxy/"

# O nome da lista tem espaços e acento; precisa ir percent-encoded no path
# (com o "í" cru a requisição nem chega a sair).
LISTA_NOTICIAS = "Multicanais - Notícias"
URL_LISTAGEM = (
    PROXY_BASE
    + "attachment/_api/web/lists/getbytitle('"
    + urllib.parse.quote(LISTA_NOTICIAS, safe="")
    + "')/items"
)
URL_DETALHE = PROXY_BASE + "api/noticias/get"

# URL pública equivalente a cada item — é essa que iria para os metadados,
# não a do endpoint interno.
URL_PAGINA_NOTICIA = "https://www.ons.org.br/paginas/noticias/details.aspx?i="

# GUID do portal ONS na taxonomia do SharePoint. Sem esse filtro a lista mistura
# notícias de outros canais (Sintegre etc).
TAXONOMIA_SITE_ONS = "902c3577-c4b1-4d76-9780-bcb001c0e0c3"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

# Origin/Referer não são decorativos: sem eles o proxy responde 405.
HEADERS_API = {
    "User-Agent": USER_AGENT,
    "Accept": "application/json;odata=verbose",
    "Content-Type": "application/json;odata=verbose",
    "Odata-Version": "3.0",
    "CacheLevel": "User",
    "Accept-Language": "pt-BR,pt;q=0.9",
    "Origin": "https://www.ons.org.br",
    "Referer": SITE_URL,
}

PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")
TAGS_LIXO = ["script", "style", "noscript", "iframe", "svg", "form", "button"]

In [0]:
def baixar_json(url: str, params: Optional[dict] = None, tentativas: int = 3) -> Optional[dict]:
    """Baixa um JSON do proxy do ONS, com httpx e fallback em curl_cffi."""
    for tentativa in range(1, tentativas + 1):
        try:
            resp = httpx.get(url, params=params, headers=HEADERS_API,
                             timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text:
                return resp.json()
            print(f"  [httpx tent {tentativa}/{tentativas}] HTTP {resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, params=params, headers=HEADERS_API,
                                     impersonate=impersonate, timeout=HTTP_TIMEOUT,
                                     allow_redirects=True)
            if resp.status_code == 200 and resp.text:
                return resp.json()
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] HTTP {resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar as notícias

Reproduz o filtro OData que o `wp_noticias.js` monta: só itens aprovados, já publicados,
não expirados e do portal ONS. A paginação é por `$skiptoken` — o campo `__next` da resposta
aponta para um host interno (`admmulticanais.ons.org.br`, inacessível de fora), então
aproveitamos só o valor do token e reemitimos contra o proxy, exatamente como o site faz.

In [0]:
def _filtro_publicadas() -> str:
    """Filtro OData equivalente ao que o site monta em m_setFilters()."""
    agora = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
    return (
        f"PublishStatus eq 'Aprovado' and "
        f"(PublishDate le datetime'{agora}' and "
        f"(ExpirationDate eq null or ExpirationDate ge datetime'{agora}')) and "
        f"TaxCatchAll/IdForTerm eq '{TAXONOMIA_SITE_ONS}'"
    )


def _extrair_skiptoken(url_next: Optional[str]) -> Optional[str]:
    """Pega só o $skiptoken do __next (o host que vem nele não é acessível de fora)."""
    if not url_next:
        return None
    query = urllib.parse.urlparse(url_next).query
    valores = urllib.parse.parse_qs(query).get("$skiptoken")
    return valores[0] if valores else None


def listar_noticias(max_itens: int = 20, por_pagina: int = 10) -> list[dict]:
    itens, skiptoken = [], None

    while len(itens) < max_itens:
        params = {
            "$select": "Id,Title,Subtitle,PublishDate",
            "$filter": _filtro_publicadas(),
            "$orderby": "PublishDate desc,Title asc",
            "$top": str(por_pagina),
        }
        if skiptoken:
            params["$skiptoken"] = skiptoken

        dados = baixar_json(URL_LISTAGEM, params=params)
        if not dados:
            print("  -> falha ao baixar a página da listagem; interrompendo.")
            break

        bloco = (dados.get("d") or {}).get("results") or []
        if not bloco:
            break

        for registro in bloco:
            publicado = registro.get("PublishDate")  # ex.: 2026-07-31T20:46:00Z
            itens.append({
                "id": registro.get("Id"),
                "titulo": registro.get("Title"),
                "subtitulo": registro.get("Subtitle"),
                "published_at": publicado[:10] if publicado else None,
                "url": f"{URL_PAGINA_NOTICIA}{registro.get('Id')}",
            })

        skiptoken = _extrair_skiptoken((dados.get("d") or {}).get("__next"))
        if not skiptoken:
            break
        time.sleep(random.uniform(0.5, 1.2))

    return itens[:max_itens]

In [0]:
noticias = listar_noticias(max_itens=15, por_pagina=10)

print(f"{len(noticias)} notícias listadas (2 páginas de 10, cortadas em 15).\n")
print(f"{'DATA':<12} {'ID':<8} TÍTULO")
print("-" * 100)
for noticia in noticias:
    print(f"{noticia['published_at'] or '?':<12} {noticia['id']:<8} {noticia['titulo'][:75]}")

# Conferências rápidas de sanidade.
ids_unicos = {n["id"] for n in noticias}
sem_data = [n for n in noticias if not n["published_at"]]
sem_titulo = [n for n in noticias if not n["titulo"]]

print(f"\nids únicos: {len(ids_unicos)}/{len(noticias)}")
print(f"sem data: {len(sem_data)} | sem título: {len(sem_titulo)}")
print(f"\nExemplo de link de detalhe: {noticias[0]['url']}")
print(f"Exemplo de subtítulo: {noticias[0]['subtitulo']}")

## Teste 2 — abrir uma notícia e extrair o texto completo

O campo `ContentSimple` do endpoint de detalhe já vem com o corpo da notícia em HTML —
basta limpar as tags. O mesmo endpoint devolve `Title`, `Subtitle` e `PublishDate`, então
não é preciso cruzar nada com a listagem.

In [0]:
def extrair_noticia(id_noticia) -> Optional[dict]:
    dados = baixar_json(URL_DETALHE, params={"id": str(id_noticia)})
    if not dados:
        return None

    html_conteudo = dados.get("ContentSimple") or ""
    try:
        soup = BeautifulSoup(html_conteudo, "lxml")
    except Exception:
        soup = BeautifulSoup(html_conteudo, "html.parser")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    texto = PADRAO_LINHAS_VAZIAS.sub("\n\n", soup.get_text("\n", strip=True)).strip()

    publicado = dados.get("PublishDate")
    return {
        "id": dados.get("ID") or id_noticia,
        "titulo": dados.get("Title"),
        "subtitulo": dados.get("Subtitle"),
        "published_at": publicado[:10] if publicado else None,
        "url": f"{URL_PAGINA_NOTICIA}{id_noticia}",
        "texto": texto,
    }

In [0]:
AMOSTRA = 5

detalhes = []
for noticia in noticias[:AMOSTRA]:
    detalhe = extrair_noticia(noticia["id"])
    if detalhe is None:
        print(f"  [FALHOU] id={noticia['id']}")
        continue
    detalhes.append(detalhe)
    time.sleep(random.uniform(0.5, 1.2))

print(f"{len(detalhes)}/{AMOSTRA} notícias abertas com sucesso.\n")
print(f"{'ID':<8} {'CHARS':<8} TÍTULO")
print("-" * 100)
for detalhe in detalhes:
    print(f"{detalhe['id']:<8} {len(detalhe['texto']):<8} {detalhe['titulo'][:70]}")

vazias = [d for d in detalhes if len(d["texto"]) < 200]
print(f"\nCom texto abaixo de 200 chars: {len(vazias)}")

In [0]:
# Amostra completa da primeira notícia — é aqui que dá pra conferir na mão se o
# texto bate com o que aparece no site.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"SUBTÍTULO   : {detalhe['subtitulo']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

In [0]:
# Como ficariam os metadados no contrato do pipeline (.json ao lado do .txt).
# Nada é gravado aqui — é só pra conferir o formato antes da Fase 3.
exemplo_metadados = {
    "source_id": "ons_noticias",
    "title": detalhe["titulo"],
    "description": "Linked from ONS — Notícias",
    "url": detalhe["url"],
    "date": datetime.now().strftime("%Y-%m-%d"),
    "published_at": detalhe["published_at"],
}

print(json.dumps(exemplo_metadados, ensure_ascii=False, indent=2))

## Conclusão da Fase 1

Os dois testes passam: a listagem devolve data + título + link para todos os itens
(ids únicos, nenhum sem data), e o texto completo sai limpo do endpoint de detalhe.

**Avaliação para a Fase 2 — não encaixa nos dispatchers genéricos.** Os três genéricos
(`ingest-news-rss-infra`, `ingest-scraping`, `ingest-PDF`) partem de "baixar uma URL e extrair
itens do HTML". Aqui o HTML é uma casca vazia: a fonte é uma API OData paginada por
`$skiptoken`, com filtro montado por data e um segundo endpoint para o corpo do texto. O
`listar()` do `ingest-scraping` recebe `(html, url_base)` — não há HTML útil para passar.

Sugestão: **notebook próprio** em `ingestores/ENERGIA/ingest-news-ons.ipynb`, com
`source_id = "ons_noticias"` (confirmar antes que não existe em `controle_fontes`) e
`atualizar_status_fonte` cobrindo os desfechos: sucesso, falha da listagem, falha no detalhe
e exceção genérica. Não precisa de Selenium, então não herda o bloqueio de Chrome/init script.

Dois pontos a decidir na Fase 3:

- **Janela de captura.** A listagem devolve o histórico inteiro por paginação. Ou limitar por
  `max_itens`, ou empurrar um `PublishDate ge` para o filtro OData e buscar só os últimos N
  dias — a segunda opção evita paginar o arquivo todo a cada execução.
- **Dedup.** O manifesto do projeto é por URL; a URL aqui é estável
  (`details.aspx?i={Id}`), então o padrão atual funciona sem adaptação.